This notebook implements the end to end data pipeline:

- resilient API ingestion into an append-only Bronze layer
- current-state Silver tables with type casting, deduplication, and `_is_valid` quality flags
- deterministic-key Gold star schema tables
- validation checks and SQL previews against the Gold layer


In [ ]:
from __future__ import annotations

import json
import math
import os
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
os.environ.setdefault("SPARK_LOCAL_HOSTNAME", "localhost")
os.environ.setdefault("PYSPARK_SUBMIT_ARGS", "--conf spark.driver.host=127.0.0.1 --conf spark.driver.bindAddress=127.0.0.1 --conf spark.ui.enabled=false pyspark-shell")
import random
import shutil
import time
from pathlib import Path

import pandas as pd
import requests
from pyspark.sql import DataFrame, SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = SparkSession.builder     .master("local[*]")     .appName("blue-owls-assessment")     .config("spark.driver.host", "127.0.0.1")     .config("spark.driver.bindAddress", "127.0.0.1")     .config("spark.ui.enabled", "false")     .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
spark.version

In [ ]:
RUN_TS = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
CUTOFF_DATE = "2018-07-01"
PAGE_SIZE = 1000
MAX_RETRIES = 6
BACKOFF_BASE_SECONDS = 1.5

cwd = Path.cwd()
SUBMISSION_DIR = cwd / "submission" if (cwd / "submission").exists() else cwd
OUTPUT_DIR = SUBMISSION_DIR / "output"
BRONZE_DIR = OUTPUT_DIR / "bronze"
SILVER_DIR = OUTPUT_DIR / "silver"
GOLD_DIR = OUTPUT_DIR / "gold"
QUARANTINE_DIR = BRONZE_DIR / "quarantine"
SQL_DIR = SUBMISSION_DIR / "sql"
MANIFEST_PATH = BRONZE_DIR / "_manifest.json"

for path in [BRONZE_DIR, SILVER_DIR, GOLD_DIR, QUARANTINE_DIR, SQL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

API_BASE_URL = os.environ.get("API_BASE_URL", "http://127.0.0.1:8000")
API_USERNAME = os.environ.get("API_USERNAME", "candidate")
API_PASSWORD = os.environ.get("API_PASSWORD", "blue-owls-2026")

ENDPOINT_CONFIG = {
    "orders": {"natural_key": ["order_id"], "params": {"date_from": CUTOFF_DATE}},
    "order_items": {"natural_key": ["order_id", "order_item_id"], "params": {"date_from": CUTOFF_DATE}},
    "customers": {"natural_key": ["customer_id"], "params": {}},
    "products": {"natural_key": ["product_id"], "params": {}},
    "sellers": {"natural_key": ["seller_id"], "params": {}},
    "payments": {"natural_key": ["order_id", "payment_sequential"], "params": {}},
}

print({
    "submission_dir": str(SUBMISSION_DIR),
    "api_base_url": API_BASE_URL,
    "cutoff_date": CUTOFF_DATE,
    "run_ts": RUN_TS,
})

In [ ]:
class APIClient:
    def __init__(self, base_url: str, username: str, password: str) -> None:
        self.base_url = base_url.rstrip("/")
        self.username = username
        self.password = password
        self.session = requests.Session()
        self.token: str | None = None
        self.auth_count = 0

    def authenticate(self) -> str:
        response = self.session.post(
            f"{self.base_url}/api/v1/auth/token",
            json={"username": self.username, "password": self.password},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        self.token = payload["access_token"]
        self.auth_count += 1
        return self.token

    def _headers(self) -> dict[str, str]:
        if not self.token:
            self.authenticate()
        return {"Authorization": f"Bearer {self.token}"}

    def get_json(self, endpoint: str, params: dict[str, object]) -> dict:
        url = f"{self.base_url}/api/v1/data/{endpoint}"
        for attempt in range(MAX_RETRIES):
            try:
                response = self.session.get(url, headers=self._headers(), params=params, timeout=120)
            except requests.exceptions.RequestException:
                sleep_seconds = BACKOFF_BASE_SECONDS * (2 ** attempt)
                time.sleep(sleep_seconds + random.uniform(0, 0.5))
                continue

            if response.status_code == 401:
                self.authenticate()
                continue

            if response.status_code in (429, 500):
                retry_after = response.headers.get("Retry-After")
                if retry_after and retry_after.isdigit():
                    sleep_seconds = float(retry_after)
                else:
                    sleep_seconds = BACKOFF_BASE_SECONDS * (2 ** attempt)
                time.sleep(sleep_seconds + random.uniform(0, 0.5))
                continue

            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict) or "data" not in payload or "pagination" not in payload:
                raise ValueError(f"Unexpected payload shape for {endpoint}: {payload}")
            if not isinstance(payload["data"], list) or not isinstance(payload["pagination"], dict):
                raise ValueError(f"Malformed payload for {endpoint}: {payload}")
            return payload

        raise RuntimeError(f"Exceeded retry limit for endpoint={endpoint}, params={params}")


def load_manifest() -> dict:
    if MANIFEST_PATH.exists():
        return json.loads(MANIFEST_PATH.read_text())
    return {"pages": {}}


def save_manifest(manifest: dict) -> None:
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True))


def append_rows_to_csv(target_path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    target_path.parent.mkdir(parents=True, exist_ok=True)
    frame = pd.DataFrame(rows)
    frame.to_csv(target_path, mode="a", index=False, header=not target_path.exists())


def build_page_key(endpoint: str, params: dict[str, object]) -> str:
    return json.dumps({"endpoint": endpoint, **params}, sort_keys=True, default=str)


def validate_record(record: object, natural_key: list[str]) -> tuple[bool, dict]:
    if not isinstance(record, dict):
        return False, {"reason": "record_is_not_a_dict", "raw_record": str(record)}
    missing_keys = [key for key in natural_key if not record.get(key)]
    if missing_keys:
        enriched = dict(record)
        enriched["_quarantine_reason"] = f"missing_natural_key:{','.join(missing_keys)}"
        return False, enriched
    return True, dict(record)


def read_bronze_csv(endpoint: str) -> pd.DataFrame:
    path = BRONZE_DIR / f"{endpoint}.csv"
    if not path.exists():
        return pd.DataFrame()
    return pd.read_csv(path)


def ingest_endpoint(
    client: APIClient,
    manifest: dict,
    endpoint: str,
) -> dict[str, int]:
    natural_key = ENDPOINT_CONFIG[endpoint]["natural_key"]
    base_params = dict(ENDPOINT_CONFIG[endpoint]["params"])
    page = 1
    pages_seen = 0
    rows_written = 0
    quarantined = 0

    while True:
        params = {**base_params, "page": page, "page_size": PAGE_SIZE}
        page_key = build_page_key(endpoint, params)
        page_state = manifest["pages"].get(page_key)

        if page_state:
            pages_seen += 1
            if not page_state["pagination"].get("has_next", False):
                break
            page += 1
            continue

        payload = client.get_json(endpoint, params)
        ingested_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        good_rows: list[dict] = []
        bad_rows: list[dict] = []

        for record in payload["data"]:
            is_valid_record, normalized = validate_record(record, natural_key)
            if not is_valid_record:
                normalized["_ingested_at"] = ingested_at
                normalized["_source_endpoint"] = endpoint
                bad_rows.append(normalized)
                continue
            normalized["_ingested_at"] = ingested_at
            normalized["_source_endpoint"] = endpoint
            good_rows.append(normalized)

        append_rows_to_csv(BRONZE_DIR / f"{endpoint}.csv", good_rows)
        append_rows_to_csv(QUARANTINE_DIR / f"{endpoint}_malformed.csv", bad_rows)

        manifest["pages"][page_key] = {
            "endpoint": endpoint,
            "params": params,
            "ingested_at": ingested_at,
            "rows_written": len(good_rows),
            "rows_quarantined": len(bad_rows),
            "pagination": payload["pagination"],
        }
        save_manifest(manifest)

        pages_seen += 1
        rows_written += len(good_rows)
        quarantined += len(bad_rows)

        if not payload["pagination"].get("has_next", False):
            break
        page += 1

    return {
        "endpoint": endpoint,
        "pages_seen": pages_seen,
        "rows_written": rows_written,
        "rows_quarantined": quarantined,
    }


client = APIClient(API_BASE_URL, API_USERNAME, API_PASSWORD)
manifest = load_manifest()

In [ ]:
ingestion_summary = []

for endpoint in ["orders", "order_items", "customers", "products", "sellers", "payments"]:
    ingestion_summary.append(ingest_endpoint(client, manifest, endpoint))

pd.DataFrame(ingestion_summary)

In [ ]:
def read_bronze_spark(endpoint: str) -> DataFrame:
    path = BRONZE_DIR / f"{endpoint}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing Bronze file for endpoint: {endpoint}")
    return spark.read.option("header", True).csv(str(path))


def latest_by_key(df: DataFrame, key_cols: list[str]) -> DataFrame:
    window = Window.partitionBy(*key_cols).orderBy(F.col("_ingested_at").desc())
    return (
        df.dropDuplicates()
          .withColumn("_rn", F.row_number().over(window))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )


def write_single_csv(df: DataFrame, target_path: Path, order_by: list[str]) -> None:
    temp_dir = target_path.parent / f"_{target_path.stem}_tmp"
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    if target_path.exists():
        target_path.unlink()

    target_path.parent.mkdir(parents=True, exist_ok=True)
    df.orderBy(*order_by).coalesce(1).write.mode("overwrite").option("header", True).csv(str(temp_dir))
    part_file = next(temp_dir.glob("part-*.csv"))
    shutil.move(str(part_file), str(target_path))
    shutil.rmtree(temp_dir)


def decimal_type() -> T.DecimalType:
    return T.DecimalType(18, 2)


raw_orders = read_bronze_spark("orders")
raw_order_items = read_bronze_spark("order_items")
raw_customers = read_bronze_spark("customers")
raw_products = read_bronze_spark("products")
raw_sellers = read_bronze_spark("sellers")
raw_payments = read_bronze_spark("payments")

In [ ]:
orders_silver = (
    raw_orders
    .withColumn("order_purchase_timestamp", F.to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at", F.to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date", F.to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date", F.to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date", F.to_timestamp("order_estimated_delivery_date"))
    .withColumn("_ingested_at", F.to_timestamp("_ingested_at"))
    .filter(F.to_date("order_purchase_timestamp") >= F.lit(CUTOFF_DATE))
    .withColumn(
        "_is_valid",
        F.col("order_id").isNotNull()
        & F.col("customer_id").isNotNull()
        & F.col("order_purchase_timestamp").isNotNull()
        & (
            F.col("order_delivered_customer_date").isNull()
            | (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))
        ),
    )
)
orders_silver = latest_by_key(orders_silver, ["order_id"])

qualifying_orders = orders_silver.select("order_id", "customer_id").distinct()
qualifying_customer_ids_df = qualifying_orders.select("customer_id").distinct()

customers_silver = (
    raw_customers
    .withColumn("_ingested_at", F.to_timestamp("_ingested_at"))
    .join(qualifying_customer_ids_df.withColumn("_is_in_scope", F.lit(True)), on="customer_id", how="inner")
    .drop("_is_in_scope")
    .withColumn("_is_valid", F.col("customer_id").isNotNull() & F.col("customer_unique_id").isNotNull())
)
customers_silver = latest_by_key(customers_silver, ["customer_id"])

order_items_silver = (
    raw_order_items
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .withColumn("price", F.col("price").cast(decimal_type()))
    .withColumn("freight_value", F.col("freight_value").cast(decimal_type()))
    .withColumn("_ingested_at", F.to_timestamp("_ingested_at"))
    .join(qualifying_orders.select("order_id").distinct().withColumn("_has_matching_order", F.lit(True)), on="order_id", how="left")
    .withColumn(
        "_is_valid",
        F.col("order_id").isNotNull()
        & F.col("order_item_id").isNotNull()
        & F.col("product_id").isNotNull()
        & F.col("seller_id").isNotNull()
        & (F.col("price").isNull() | (F.col("price") >= F.lit(0)))
        & (F.col("freight_value").isNull() | (F.col("freight_value") >= F.lit(0)))
        & F.coalesce(F.col("_has_matching_order"), F.lit(False))
    )
    .drop("_has_matching_order")
)
order_items_silver = latest_by_key(order_items_silver, ["order_id", "order_item_id"])

qualifying_product_ids_df = order_items_silver.filter(F.col("_is_valid")).select("product_id").distinct()
qualifying_seller_ids_df = order_items_silver.filter(F.col("_is_valid")).select("seller_id").distinct()
qualifying_payment_order_ids_df = qualifying_orders.select("order_id").distinct()

products_silver = (
    raw_products
    .withColumn("product_name_lenght", F.col("product_name_lenght").cast("int"))
    .withColumn("product_description_lenght", F.col("product_description_lenght").cast("int"))
    .withColumn("product_photos_qty", F.col("product_photos_qty").cast("int"))
    .withColumn("product_weight_g", F.col("product_weight_g").cast(decimal_type()))
    .withColumn("product_length_cm", F.col("product_length_cm").cast(decimal_type()))
    .withColumn("product_height_cm", F.col("product_height_cm").cast(decimal_type()))
    .withColumn("product_width_cm", F.col("product_width_cm").cast(decimal_type()))
    .withColumn("_ingested_at", F.to_timestamp("_ingested_at"))
    .join(qualifying_product_ids_df.withColumn("_is_in_scope", F.lit(True)), on="product_id", how="inner")
    .drop("_is_in_scope")
    .withColumn("product_category_name", F.coalesce(F.col("product_category_name"), F.lit("unknown")))
    .withColumn("_is_valid", F.col("product_id").isNotNull())
)
products_silver = latest_by_key(products_silver, ["product_id"])

sellers_silver = (
    raw_sellers
    .withColumn("_ingested_at", F.to_timestamp("_ingested_at"))
    .join(qualifying_seller_ids_df.withColumn("_is_in_scope", F.lit(True)), on="seller_id", how="inner")
    .drop("_is_in_scope")
    .withColumn("_is_valid", F.col("seller_id").isNotNull())
)
sellers_silver = latest_by_key(sellers_silver, ["seller_id"])

payments_silver = (
    raw_payments
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast(decimal_type()))
    .withColumn("_ingested_at", F.to_timestamp("_ingested_at"))
    .join(qualifying_payment_order_ids_df.withColumn("_is_in_scope", F.lit(True)), on="order_id", how="inner")
    .drop("_is_in_scope")
    .withColumn(
        "_is_valid",
        F.col("order_id").isNotNull()
        & F.col("payment_sequential").isNotNull()
        & (F.col("payment_value").isNull() | (F.col("payment_value") >= F.lit(0)))
    )
)
payments_silver = latest_by_key(payments_silver, ["order_id", "payment_sequential"])

silver_tables = {
    "orders": orders_silver,
    "order_items": order_items_silver,
    "customers": customers_silver,
    "products": products_silver,
    "sellers": sellers_silver,
    "payments": payments_silver,
}

for table_name, df in silver_tables.items():
    write_single_csv(df, SILVER_DIR / f"{table_name}.csv", [col for col in df.columns if col in {"order_id", "order_item_id", "customer_id", "product_id", "seller_id"} or col.endswith("_id")][:3] or df.columns[:1])

{k: v.count() for k, v in silver_tables.items()}

In [ ]:
orders_valid = orders_silver.filter(F.col("_is_valid"))
order_items_valid = order_items_silver.filter(F.col("_is_valid"))
customers_valid = customers_silver.filter(F.col("_is_valid"))
products_valid = products_silver.filter(F.col("_is_valid"))
sellers_valid = sellers_silver.filter(F.col("_is_valid"))
payments_valid = payments_silver.filter(F.col("_is_valid"))

customer_order_details = (
    orders_valid.alias("o")
    .join(customers_valid.alias("c"), on="customer_id", how="inner")
    .filter(F.col("c.customer_unique_id").isNotNull())
    .select(
        F.col("c.customer_unique_id"),
        F.col("c.customer_city"),
        F.col("c.customer_state"),
        F.col("c.customer_zip_code_prefix"),
        F.col("o.order_id"),
        F.col("o.order_purchase_timestamp"),
    )
)

customer_stats = (
    customer_order_details
    .groupBy("customer_unique_id")
    .agg(
        F.min(F.to_date("order_purchase_timestamp")).alias("first_order_date"),
        F.countDistinct("order_id").alias("total_orders"),
    )
)

customer_latest_window = Window.partitionBy("customer_unique_id").orderBy(F.col("order_purchase_timestamp").desc(), F.col("order_id").desc())
customer_latest = (
    customer_order_details
    .withColumn("_rn", F.row_number().over(customer_latest_window))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "order_id", "order_purchase_timestamp")
)

dim_customers = (
    customer_latest
    .join(customer_stats, on="customer_unique_id", how="inner")
    .withColumn("customer_key", F.crc32(F.col("customer_unique_id")))
    .select(
        "customer_key",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
        "first_order_date",
        "total_orders",
    )
)

dim_products = (
    products_valid
    .withColumn("product_key", F.crc32(F.col("product_id")))
    .withColumn(
        "product_volume_cm3",
        F.when(
            F.col("product_length_cm").isNull() | F.col("product_height_cm").isNull() | F.col("product_width_cm").isNull(),
            F.lit(None),
        ).otherwise(F.col("product_length_cm") * F.col("product_height_cm") * F.col("product_width_cm"))
    )
    .select(
        "product_key",
        "product_id",
        "product_category_name",
        "product_weight_g",
        F.col("product_volume_cm3").cast(decimal_type()).alias("product_volume_cm3"),
        "product_photos_qty",
        F.col("product_description_lenght").alias("product_description_length"),
    )
)

dim_sellers = (
    sellers_valid
    .withColumn("seller_key", F.crc32(F.col("seller_id")))
    .select(
        "seller_key",
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix",
    )
)

payment_choice_window = Window.partitionBy("order_id").orderBy(F.col("payment_value").desc(), F.col("payment_type").asc())
payment_primary = (
    payments_valid
    .withColumn("_rn", F.row_number().over(payment_choice_window))
    .filter(F.col("_rn") == 1)
    .select("order_id", F.col("payment_type").alias("primary_payment_type"))
)

payments_by_order = (
    payments_valid
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").cast(decimal_type()).alias("total_payment_value"),
        F.max("payment_installments").alias("max_payment_installments"),
    )
    .join(payment_primary, on="order_id", how="left")
)

order_price_totals = (
    order_items_valid.groupBy("order_id").agg(F.sum("price").alias("order_total_item_price"))
)

fact_order_items = (
    order_items_valid.alias("oi")
    .join(orders_valid.alias("o"), on="order_id", how="inner")
    .join(customers_valid.alias("c"), on="customer_id", how="inner")
    .join(dim_customers.alias("dc"), on="customer_unique_id", how="inner")
    .join(dim_products.alias("dp"), on="product_id", how="inner")
    .join(dim_sellers.alias("ds"), on="seller_id", how="inner")
    .join(payments_by_order.alias("po"), on="order_id", how="left")
    .join(order_price_totals.alias("pt"), on="order_id", how="left")
    .withColumn("order_item_sk", F.crc32(F.concat_ws("||", F.col("order_id"), F.col("order_item_id").cast("string"))))
    .withColumn("order_date", F.to_date("order_purchase_timestamp"))
    .withColumn(
        "payment_value",
        F.when(
            F.col("order_total_item_price").isNull() | (F.col("order_total_item_price") == 0) | F.col("total_payment_value").isNull(),
            F.lit(None),
        ).otherwise((F.col("total_payment_value") * F.col("price") / F.col("order_total_item_price")).cast(decimal_type()))
    )
    .withColumn(
        "days_to_deliver",
        F.when(
            F.col("order_delivered_customer_date").isNull(),
            F.lit(None),
        ).otherwise(F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_purchase_timestamp")))
    )
    .withColumn(
        "days_delivery_vs_estimate",
        F.when(
            F.col("order_delivered_customer_date").isNull() | F.col("order_estimated_delivery_date").isNull(),
            F.lit(None),
        ).otherwise(F.datediff(F.to_date("order_delivered_customer_date"), F.to_date("order_estimated_delivery_date")))
    )
    .withColumn(
        "is_late_delivery",
        F.when(F.col("order_delivered_customer_date").isNull(), F.lit(None))
         .otherwise(F.col("days_delivery_vs_estimate") > 0)
    )
    .select(
        "order_item_sk",
        "order_id",
        "order_item_id",
        F.col("customer_key").alias("customer_key"),
        F.col("product_key").alias("product_key"),
        F.col("seller_key").alias("seller_key"),
        "order_date",
        "order_status",
        F.col("price").cast(decimal_type()).alias("price"),
        F.col("freight_value").cast(decimal_type()).alias("freight_value"),
        F.col("payment_value").cast(decimal_type()).alias("payment_value"),
        F.col("primary_payment_type").alias("payment_type"),
        F.col("max_payment_installments").alias("payment_installments"),
        "days_to_deliver",
        "days_delivery_vs_estimate",
        "is_late_delivery",
    )
)

gold_tables = {
    "dim_customers": dim_customers,
    "dim_products": dim_products,
    "dim_sellers": dim_sellers,
    "fact_order_items": fact_order_items,
}

for table_name, df in gold_tables.items():
    write_single_csv(df, GOLD_DIR / f"{table_name}.csv", [df.columns[0]])

for table_name, df in gold_tables.items():
    df.createOrReplaceTempView(table_name)

record_counts = {table_name: df.count() for table_name, df in gold_tables.items()}
customer_orphans = fact_order_items.join(dim_customers.select("customer_key"), on="customer_key", how="left_anti").count()
product_orphans = fact_order_items.join(dim_products.select("product_key"), on="product_key", how="left_anti").count()
seller_orphans = fact_order_items.join(dim_sellers.select("seller_key"), on="seller_key", how="left_anti").count()

print("Record counts:", record_counts)
print({
    "customer_key_orphans": customer_orphans,
    "product_key_orphans": product_orphans,
    "seller_key_orphans": seller_orphans,
    "auth_calls": client.auth_count,
})

In [ ]:
query_1_sql = (SQL_DIR / "query_1.sql").read_text()
query_2_sql = (SQL_DIR / "query_2.sql").read_text()

print("Query 1 preview")
spark.sql(query_1_sql).show(10, truncate=False)

print("Query 2 preview")
spark.sql(query_2_sql).show(10, truncate=False)

## Final

- Quarantined malformed Bronze records, if any, are written under `output/bronze/quarantine/`.
